In [4]:
import sys

import numpy as np
import joblib
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '../../..') 
from utils.constants import Configs
from utils.dtw_radius import load_dtw_matrices

TARGET_MOVEMENT = 'Move'     # then repeat for 'Move'

In [ ]:
# DTW: all five radii (no need for selected_radius)
matrices = load_dtw_matrices(TARGET_MOVEMENT, Configs.DTW_RADIUS_Seconds,
                             Configs.wrist_only_constrained_dtw_distance_matrix)

# Features
features_df = joblib.load(Configs.wrist_only_feature_extraction_features(TARGET_MOVEMENT))
features_df = features_df.sort_values(by=['sub', 'repID', 'movID']).reset_index(drop=True)
X_features = features_df[Configs.FEATURE_COLS].to_numpy()

def nn_same_vs_other(D, subjects):
    D = np.array(D, dtype=float)
    np.fill_diagonal(D, np.inf)
    s = np.asarray(subjects)
    nn_all = D.min(axis=1)
    nn_other = np.array([D[i][s != s[i]].min() for i in range(len(D))])
    return nn_all.mean(), nn_other.mean(), (nn_other / nn_all).mean()

subs = features_df['sub'].values
for w, D in matrices.items():
    assert len(D) == len(subs), "DTW matrix and features have different rows"
    print(f"DTW {w:.1f}s :", [round(v, 3) for v in nn_same_vs_other(D, subs)])

Xs = StandardScaler().fit_transform(X_features)
print("Features :", [round(v, 3) for v in nn_same_vs_other(pairwise_distances(Xs), subs)])

DTW 0.1s : [np.float64(305.868), np.float64(500.806), np.float64(1.927)]
DTW 0.2s : [np.float64(286.239), np.float64(488.332), np.float64(2.018)]
DTW 0.3s : [np.float64(280.145), np.float64(481.929), np.float64(2.017)]
DTW 0.4s : [np.float64(277.992), np.float64(479.454), np.float64(2.012)]
DTW 0.5s : [np.float64(276.92), np.float64(478.256), np.float64(2.011)]
Features : [np.float64(2.527), np.float64(2.665), np.float64(1.068)]
